# prepare GPUs

In [1]:
import os

# nvidia-smi

# Limit PyTorch to use specific GPUs
os.environ["CUDA_VISIBLE_DEVICES"] ="0,1,2,3" # "2,3,4,5"  # 

import torch

torch.cuda.init()  

# Check available GPUs
print("Available GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_ids =[0,1,2,3]  # [2,3,4,5]  # used in sequence detector for parallel
print('Using device = ', device)

Available GPUs: 4
GPU 0: NVIDIA H100 80GB HBM3
GPU 1: NVIDIA H100 80GB HBM3
GPU 2: NVIDIA H100 80GB HBM3
GPU 3: NVIDIA H100 80GB HBM3
Using device =  cuda


In [2]:

import gc
gc.collect()
torch.cuda.empty_cache()

# Print memory allocated for each GPU
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Allocated: {torch.cuda.memory_allocated(i)/1024**2:.2f} MB")
    print(f"  Cached:    {torch.cuda.memory_reserved(i)/1024**2:.2f} MB")
    
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) 


GPU 0: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 1: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 2: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
GPU 3: NVIDIA H100 80GB HBM3
  Allocated: 0.00 MB
  Cached:    0.00 MB
2.5.1
True
NVIDIA H100 80GB HBM3


In [3]:

import numpy as np
import sys
sys.path.append('..')

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import random
# from collections import Counter
import json

# DeepCASE Imports
from deepcase.preprocessing   import Preprocessor
from deepcase.context_builder import ContextBuilder
from deepcase.interpreter     import Interpreter


In [4]:
from AIC.utils import *

In [5]:
# for BERT
from sklearn.metrics import precision_recall_fscore_support # , f1_score
from sklearn.preprocessing import MinMaxScaler
import torch.nn.functional as F
from tqdm import tqdm
from model import Model
from log_attention import LogAttention
from dataloader import DataGenerator

import torch.nn as nn
import torch.optim as optim
import time


# reconstruct data if it is raw from OpTC
do need do this module if data has existed in the folder "PredataBERT"

In [ ]:
'''convert json to csv & align timestamp'''

# 18-23 Sep
# input_folder = "../OpTC-part/ecar-bro-benign-hosts-5days"  # json files
# output_folder = "../OpTC-part/ecar-bro-benign-hosts-5days-csv"
# 19 Sep
# input_folder = "../OpTC-part/ecar-bro-benign-hosts-19Sep19"  # json files
# output_folder = "../OpTC-part/ecar-bro-benign-hosts-19Sep19-csv"

# new files
input_folder = "../OpTC-part/BenignAnomalyHost"  # json files
output_folder = "../OpTC-part/BenignAnomalyHost-csv"

# Loop through all JSON files in the input folder
json2csv_timestamp(input_folder,output_folder)

In [ ]:
'''assign fields and values for BERT'''

# file_folder="../OpTC-part/ecar-bro-benign-hosts-5days-csv"
# file_folder="../OpTC-part/ecar-bro-benign-hosts-19Sep19-csv"
# file_folder="../OpTC-part/BenignAnomalyHost-csv"
file_folder="../OpTC-part/temp"

# extract_folder='./PredataBERT'
extract_folder='./PreAnomalydataBERT'

short_file='shortBERT.csv'

reConstructData( file_folder,extract_folder,short_file)


In [ ]:
''' save event index and event type name in 'shortBERTIndex.csv' '''
readname='shortBERT.csv'
addIndex4Uniq(readname)

In [ ]:
print(labelset.loc[:,'hostname'].unique())

In [ ]:
# # on-line update event type, short csv file
# def update_short(shortname):
#     short_old=pd.read_csv(short_file)
#     if shortname not in short_old['event'] :
#         new_row={"id":len(short_old), "event":shortname}
#         short_old.loc[len(short_old)]=new_row
#         short_old.to_csv(short_file, index=False)
    

In [ ]:
'''there is no duplicated records here through testing. for OpTC'''
# find the index of duplicated records.
data_folder = "../OpTC-part/ecar-bro-benign-hosts-19Sep19-csv"

# Loop through all JSON files in the input folder
for filename in os.listdir(data_folder):        
    data_temp = pd.read_csv(os.path.join(data_folder, filename))
    print(filename)
    print(len(data_temp))
    dup_bool=data_temp.duplicated(subset=['timestamp','hostname','id'])   #  ['time','ip','short'] (short is the attack type) change to : ['timestamp','hostname','id']
    dup_indexs=np.where(dup_bool==False)[0]    
    print(len(dup_indexs))

In [ ]:
#drop duplicated records and store in new files.
for filename in os.listdir(data_folder):        
    data_temp = pd.read_csv(filename)
    print(filename)
    print(len(data_temp))
    data_temp = data_temp.drop_duplicates(subset=['timestamp','hostname','id']) 
    # Save DataFrame to a text file
    new_name=filename.split('.csv')[0]+'_uniq'+'.csv'
    data_temp.to_csv(new_name, index=False) 
    print("DataFrame saved to "+new_name)

# params prepare

In [6]:
import argparse
from argparse import Namespace

def save_args(args, path="saved_model/args.json"):
    """Save argparse.Namespace to a JSON file"""
    with open(path, "w") as f:
        json.dump(vars(args), f, indent=2)
        
def load_args(path="saved_model/args.json"):
    """Load argparse.Namespace from a JSON file"""
    with open(path, "r") as f:
        args_dict = json.load(f)
    return Namespace(**args_dict)

In [7]:
# if load from existed args
# args=load_args(path="saved_model/args-8619.json")
args=load_args()

In [7]:
# create a new args
args=Namespace(
    mode=   'classifier', # sequence detector without log parameters encoder using Transformer with multiheadattenion
#     'paramEncode', # concern log parameters encoder in Transformer instead of multiheadattention
    events='auto',  #help="number of distinct events to handle"
    length=10, # help="sequence LENGTH, window size")
    timeout=86400, #help="sequence TIMEOUT (seconds)")
    # save_sequences=None,  # help="path to save sequences")
    # load_sequences=None, # help="path to load sequences")

########################### Add ContextBuilder arguments
    hidden=128, # help="HIDDEN layers dimension")
    delta=0.1,  # help="label smoothing DELTA")
    save_builder='saved_model/saved_CB_model.pth',  #help="path to save ContextBuilder")
    load_builder=None, # help="path to load ContextBuilder")

######################## Add Training arguments for ContextBuilder
    epochs_CB= 10, # 50 , # help="number of epochs to train with")
    batch_CB= 4096, # 128, #help="batch size       to train with")

####################### Add Interpreter arguments
    confidence=0.1,  # help="minimum required CONFIDENCE")
    epsilon=0.1,  # help="DBSCAN clustering EPSILON")
    min_samples=5  , # help="DBSCAN clustering MIN_SAMPLES")
    save_interpreter='saved_model/saved_Interpreter_model.pth', #  help="path to save Interpreter")
    load_interpreter=None, #  help="path to load Interpreter")
# group_interpreter.add_argument('--save-clusters'   , help="path to CSV file to save clusters")
# group_interpreter.add_argument('--load-clusters'   , help="path to CSV file to load clusters")
    save_prediction=None, # help="path to CSV file to save prediction")




#################### Add SequenceDetector arguments
    resume=0, #help="resume training of model (0/no, 1/yes)")
    load_SequenceDetecter='checkpoints/model-latest.pt', # help="latest model path")
    # temp_best_f1=0,
    
#################### Add Training arguments for Sequence Detector
    epochs_SD=20, # 50 , # help="number of epochs to train with")
    batch_SD=12*10, # help="batch size to train with, **** must be****  divided by window size.")
    lr_SD=0.0001,
    num_layers_SD=1, # help='num of encoder layer')
    threshold_SD=0.5, # help='threhold value for evaluation')
    embed_content_dim=768,
    embed_param_dim=3,
    embed_context_dim=128,
    alpha_loss=0.5,
    
#################### Add DataLoader arguments
    dataloader_batch_size=1280,
    dataloader_chunk_size=400000,

####################### Add other arguments
    device='auto' , #  help="DEVICE used for computation (cpu|cuda|auto)")
    silent=True, # action='store_true', help="silence mode, do not print progress")

# useless when concern file split
    train_test_size=0.1,   # help="split all data into training set and test set")
    train_validate_size=0.2,  # help="split training set into train set and validate set")

)

# DataPreprocessing


In [8]:
########################################################################
#                             Loading data                             #
########################################################################

# Create preprocessor
preprocessor = Preprocessor(
    length  = args.length,    # 10 events in context, also window size in Sequence Detector
    timeout = args.timeout,  #  86400, # Ignore events older than 1 day (60*60*24 = 86400 seconds)
)
    

In [9]:
# load data from files for Interpreter, cluster set
file_folder = "PredataBERT"
test_filename = "18-19-053.csv"

# # if Process all CSV files in the folder
# frame_list = []
# for file_name in os.listdir(file_folder):    
#     if file_name==test_filename:  # file_name.endswith(".csv"):
#         file_path = os.path.join(file_folder, file_name)  
#         data_temp=pd.read_csv(file_path, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]
#         # ############ temperaly get 10000 records
#         frame_list.append(data_temp)   # .loc[:1000000,:]) 
#         print("read file:"+ file_path+" , len="+ str(len(data_temp)) )
# data = pd.concat(frame_list)
### data =preprocessor.read_csv_files( test_data)   #  read four necessary columns from all csv files
# data.reset_index(drop=True, inplace=True)

# del data_temp
# del frame_list
# torch.cuda.empty_cache() 


file_path= os.path.join(file_folder, test_filename) 

''' all training data files.
AIA-51-75.ecar-last.csv, AIA-101-125.ecar-last.csv
AIA-151-175.ecar-2019-12-07T17-24-50.643.csv, AIA-151-175.ecar.csv, 
AIA-201-225.ecar-2019-12-07T16-16-05.667.csv, AIA-201-225.ecar.csv
'''
''' 18-19-051.csv,(8646),18-19-052.csv,(8745), 18-19-053.csv,  18-19-054.csv,18-19-055.csv
19-051.csv, 19-052.csv, 19-053.csv, 19-054.csv, 19-055.csv
20-23-051.csv, 20-23-052.csv(8858), 20-23-053.csv, 20-23-054.csv,20-23-055.csv (0.17)
'''
data=pd.read_csv(file_path, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]

data = data.rename(columns = {"hostname":"machine"})
data['label']=0 # label for each event. not work on the performance 


In [10]:
''' preprocess for ContextBuilder'''

mapping=get_event_mapping("shortBERTIndex.csv")

start_time = time.time()
# option2
# context, events, mapping_label, labels=preprocessor.sequence(data,labels=None,verbose=False, existed_mapping=mapping)
# option1
context_train, events_train, mapping_label, labels_train_binary=preprocessor.sequence(data,labels=None,verbose=False, existed_mapping=mapping)
time_cost = time.time()-start_time
print(f"Time cost: {time_cost:.3f} seconds")

# print(f'the total number of samples: {len(labels)}')
# print(f'the number of false positive: {sum(labels==0)}')
# print(f'the number of analomal samples: {sum(labels==1)} ')


# Automatically set device argument
if args.device == "auto":
    args.device = "cuda" if torch.cuda.is_available() else "cpu"

# Automatically set the number of events to expect
if args.events == "auto":
    args.events = len(mapping)
else:
    args.events = int(args.events)

Time cost: 269.750 seconds


In [11]:
# Cast tensors to device
events_train  = events_train.to(args.device)
context_train = context_train.to(args.device)

## specific test file data

In [12]:

test_file ="PreAnomalydataBERT/20-23-124(20).csv"
label_file=("../OpTC-part/BenignAnomalyHost-csv/20-23-124(20)_labeled.csv")
            
data_test=pd.read_csv(test_file, usecols=['id','timestamp','hostname','event','field_names','field_values'])[['id','timestamp','hostname','event','field_names','field_values']]
data_test = data_test.rename(columns = {"hostname":"machine"})

data_test['label']=pd.read_csv(label_file, usecols=["label"]) 


In [13]:
start_time = time.time()
context_test, events_test, _, labels_test_binary=preprocessor.sequence(data_test,labels=None,verbose=False, existed_mapping=mapping)
time_cost = time.time()-start_time
print(time_cost)

print(f'the total number of samples: {len(labels_test_binary)}')
print(f'the number of false positive: {sum(labels_test_binary==0)}')
print(f'the number of analomal samples: {sum(labels_test_binary==1)} ')

123.11370420455933
the total number of samples: 2775767
the number of false positive: 2775169
the number of analomal samples: 598 


In [14]:

frames=[pd.DataFrame(context_test.detach().numpy()), data_test]  # 
X_test = pd.concat(frames, axis=1)  


In [15]:
# Cast tensors to device
events_test  = events_test.to(args.device)
context_test = context_test.to(args.device)

# ContextBuilder
when need to fit the Context Builder

In [16]:
########################################################################
#                         Using ContextBuilder                         #
########################################################################

# args.load_builder= "saved_model/saved_CB_model-2.pth"    #   
# args.load_builder= "saved_model/saved_CB_model-2-8619.pth"
# args.load_builder=  None   #
args.load_builder= "saved_model/saved_CB_model-2-2.pth"
# Load the builder, if necessary
if args.load_builder:
    context_builder = ContextBuilder.load(args.load_builder, args.device)

# Otherwise create a new ContextBuilder
else:
    # Create ContextBuilder
    context_builder = ContextBuilder(
        input_size    = args.events,
        output_size   = args.events,
        hidden_size   = args.hidden,
        num_layers    = 1,
        max_length    = args.length,
        bidirectional = False,
        LSTM          = False,
    ).to(args.device)


/home/tengfei/MyCode/AIC/deepcase/context_builder/context_builder.py:650: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(infile, map_location=device)


In [17]:
def process_large_data_CB_fit(big_contexts, big_events,  big_labels, args,model=None):
    """
    Process a large dataset in chunks and combine the results.
    
    Args:
        features: Large feature tensor/array
        labels: Labels tensor/array
        batch_size: Batch size for DataLoader
        chunk_size: Size of each chunk to process
        model: PyTorch model to use for inference
        device: Device to run the model on ('cuda' or 'cpu')
        args: Namespace object containing parameters
    Returns:
        Combined output from processing all chunks
    """
    # batch_size=args.dataloader_batch_size
    chunk_size=args.dataloader_chunk_size *3 # // 4
    
    # Get the total size of the dataset
    total_size = len(big_events)
    
    # Calculate the number of chunks
    num_chunks = (total_size + chunk_size - 1) // chunk_size
        
    # Process each chunk
    for i in range(num_chunks):
        # Calculate start and end indices for this chunk
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total_size)
        
        print(f"Processing chunk {i+1}/{num_chunks} (indices {start_idx} to {end_idx})")
        
        # Get subset of features and labels for this chunk
        chunk_contexts = big_contexts[start_idx:end_idx]
        chunk_events = big_events[start_idx:end_idx]
        chunk_labels = big_labels[start_idx:end_idx]
        
        # # Create a DataGenerator and DataLoader for this chunk
        # chunk_generator = DataGenerator(chunk_features, chunk_labels)
        # chunk_loader = torch.utils.data.DataLoader(chunk_generator, batch_size=batch_size, shuffle=False)

        # Train the ContextBuilder
        model.fit(
            X             = chunk_contexts,               # Context to train with
            y             = chunk_events.reshape(-1, 1), # Events to train with, note that these should be of shape=(n_events, 1)
            labels        = chunk_labels,
            epochs        = args.epochs_CB,                          # Number of epochs to train with
            batch_size    = args.batch_CB,                         # Number of samples in each training batch, in paper this was 128
            learning_rate = 0.001,                        # Learning rate to train with, in paper this was 0.01
            verbose       =  args.silent, # not args.silent,                        # If True, prints progress
            optimizer= optim.SGD,
            teach_ratio=0.5
        )

    return model
       

In [18]:
import tempfile

def process_large_data_CB_predict(big_contexts, big_events, args, model=None):
    chunk_size = args.dataloader_chunk_size    // 2
    total_size = len(big_events)
    num_chunks = (total_size + chunk_size - 1) // chunk_size

    temp_dir = tempfile.mkdtemp()  # Creates a temporary directory
    result1_paths, result2_paths = [], []

    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total_size)
        print(f"Processing chunk {i+1}/{num_chunks} (indices {start_idx} to {end_idx})")

        chunk_contexts = big_contexts[start_idx:end_idx]
        chunk_events = big_events[start_idx:end_idx]

        with torch.no_grad():
            chunk_confi, chunk_atten = model.predict(chunk_contexts, chunk_events.reshape(-1, 1))

        # Save chunk results as .pt files
        path1 = os.path.join(temp_dir, f"chunk_confi_{i}.pt")
        path2 = os.path.join(temp_dir, f"chunk_atten_{i}.pt")
        torch.save(chunk_confi.cpu(), path1)
        torch.save(chunk_atten.cpu(), path2)
        result1_paths.append(path1)
        result2_paths.append(path2)

        # Clear memory
        del chunk_contexts, chunk_events, chunk_confi, chunk_atten
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Load and concatenate all results
    all_result1 = [torch.load(p, weights_only=True) for p in result1_paths]
    all_result2 = [torch.load(p, weights_only=True) for p in result2_paths]

    final_result1 = torch.cat(all_result1, dim=0)
    final_result2 = torch.cat(all_result2, dim=0)

    # Clean up temporary files
    for p in result1_paths + result2_paths:
        os.remove(p)
    os.rmdir(temp_dir)

    return final_result1, final_result2


In [ ]:
# when need to fit the Context Builder
if len(events_train)>args.dataloader_chunk_size * 3:
    context_builder=process_large_data_CB_fit(context_train, events_train,labels_train_binary,args,context_builder)
    # context_builder=process_large_data_CB(context_val, events_val,labels_val_binary,args,context_builder)
    
else:       
    # Train the ContextBuilder
    context_builder.fit(
        X             = context_train,               # Context to train with
        y             = events_train.reshape(-1, 1), # Events to train with, note that these should be of shape=(n_events, 1)
        labels        = labels_train_binary,
        epochs        = args.epochs_CB,                          # Number of epochs to train with
        batch_size    = args.batch_CB,                         # Number of samples in each training batch, in paper this was 128
        learning_rate = 0.001,                        # Learning rate to train with, in paper this was 0.01
        verbose       =  args.silent, #  not                       # If True, prints progress
        optimizer= optim.SGD,
        teach_ratio=0.5
    )

    


In [ ]:
# args.save_builder=None 
args.save_builder="saved_model/saved_CB_model-2-2.pth"
# "saved_model/saved_CB_model-2.pth"
###### Save the builder, if necessary
if args.save_builder:
    context_builder.save(args.save_builder)

In [20]:
# del data
torch.cuda.empty_cache()  # Clears unused memory
torch.cuda.ipc_collect()  # Helps reclaim fragmented memory


In [22]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True


# catch the footprint

In [ ]:
# load args params
args = load_args()


In [ ]:
# args.save_builder="saved_model/saved_CB_model-2.pth"
# args.load_builder="saved_model/saved_CB_model-2.pth"
args.load_builder="saved_model/saved_CB_model-2-2.pth"
context_builder = ContextBuilder.load(args.load_builder, args.device)



In [ ]:

f_v=X_test['field_values']
unique_values = np.unique(f_v)  # Get unique values
param_v_size=len(unique_values)
print(param_v_size)

In [ ]:
event_template= pd.DataFrame( {'EventTemplate': X_test['field_names'].drop_duplicates(ignore_index=True)})
# event_template= pd.DataFrame( {'EventTemplate': data['field_names'].drop_duplicates(ignore_index=True)})

########### to prepare  data
_, attention_t = context_builder.predict(context_test,events_test.reshape(-1, 1))
attention_t=attention_t.squeeze(1)

fieldname_em,fieldva_ind=field_name_value_prepare(origdata_fieldname=X_test['field_names'], origdata_fieldvalue=X_test['field_values'],  embmodel=embmodel, event_template=event_template)
attention_t=attention_t.to(fieldname_em.device)
context_test=context_test.to(fieldname_em.device)

# combination of field name, field value, attention vector 
te_bert=torch.cat([fieldname_em,fieldva_ind,attention_t,context_test],dim=1)
print(te_bert.size())

In [ ]:
del f_v, unique_values, fieldname_em,fieldva_ind,attention_t # , event_template
torch.cuda.empty_cache() 

In [ ]:
''' due to large dataset, use DataLoader to split data, in case of out of memory'''
test_generator = DataGenerator( te_bert, labels_test_binary)
test_loader = torch.utils.data.DataLoader(
    test_generator, batch_size=1280, shuffle=False, num_workers=4)

<!-- 
1. data includes: training data, validate data, testing data
2. ContextBuilder, fit & predict on (training data), (validate data)
   TransformerModel, fit & predict on (training data), (validate data)
3.  ContextBuilder predict on testing data
    TransformerModel predict on testing data
4. based on trained ContextBuilder and TransformerModel, to produce the labels, 
[combine context sequences to labels,] work on testing data or new data.
5. Cluster in Interpreter
6. LLM
 -->

In [ ]:
transformermodel.eval()
transformermodel.module.update_param_vocab(param_v_size)


y_true=[]
en_list, ex_list, re_en_list, re_ex_list=[],[],[],[]
for batch_idx, (xx,y) in enumerate(tqdm(test_loader)): 
    content_true=xx[:,:args.embed_content_dim]
    context_true=xx[:,-args.length:]
    x=xx[:,:-args.length]
    x = x.to(device, dtype=torch.float32)
    # y = y.to(device, dtype=torch.float32)

    recon_content, recon_context = transformermodel(x)  #.cpu()
    y_true.append(y)
    en_list.append(content_true)
    ex_list.append(context_true)
    re_en_list.append(recon_content)
    re_ex_list.append(recon_context)
        
re_en_list = torch.cat(re_en_list).cpu()
re_ex_list = torch.cat(re_ex_list).cpu()
en_list=torch.cat(en_list)
ex_list=torch.cat(ex_list)
y_true=torch.cat(y_true)

content_loss = F.mse_loss(re_en_list, en_list)
context_loss = F.cross_entropy(re_ex_list.reshape(-1, re_ex_list.size(-1)), ex_list.reshape(-1), reduction='none')
loss= alpha* context_loss+ (1-alpha) *content_loss    
loss = loss.reshape(ex_list.size(0), ex_list.size(1))  # [seq_len, window_size]
test_seq_scores=loss.mean(dim=1).reshape(-1,1)
# Initialize the MinMaxScaler
scaler = MinMaxScaler()    
test_seq_scores = scaler.fit_transform(test_seq_scores)

test_seq_labels=(test_seq_scores > args.threshold_SD).astype(float)

labels_test_binary=labels_test_binary

# # Convert to NumPy
# scores_np = test_seq_scores.detach().cpu().numpy()


In [ ]:
del y_true, en_list, ex_list, re_en_list, re_ex_list
del recon_content, recon_context, content_loss, context_loss, loss,scaler
del test_generator, test_loader
torch.cuda.empty_cache() 

In [ ]:
# --- Plot ---
plt.figure(figsize=(10, 4))
sns.lineplot(x=np.arange(len(scores_np)), y=scores_np, marker="o")
plt.title("Anomaly Scores Per Log Sequence (Autoencoder Reconstruction)")
plt.xlabel("Sequence Index")
plt.ylabel("Reconstruction Loss (Anomaly Score)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
def compute_metrics(sequence_item_labels, predicted_sequence_labels):
    assert len(sequence_item_labels) == len(predicted_sequence_labels)

    # Compute accuracy
    correct = sum(p == t for p, t in zip(predicted_sequence_labels, sequence_item_labels))
    accuracy = correct / len(sequence_item_labels)

    # Compute coverage of abnormal events
    total_abnormal_events = 0
    detected_abnormal_events = 0
    for seq_items, pred_seq_label in zip(sequence_item_labels, predicted_sequence_labels):
        for item in seq_items:
            if item == 1:
                total_abnormal_events += 1
                if pred_seq_label == 1:
                    detected_abnormal_events += 1
    coverage = detected_abnormal_events / total_abnormal_events if total_abnormal_events > 0 else 0.0

    return accuracy, coverage

accuracy, coverage = compute_metrics(sequence_item_labels, predicted_sequence_labels)
print(f"Sequence Detection Accuracy: {accuracy:.2f}")
print(f"Abnormal Event Coverage: {coverage:.2f}")


# cluster in Interpreter
cluster on training data.
predict the score of cluster which the most similar sequence in. otherwise the predict label is -2, means no label data in training.
result : np.array of shape=(n_samples,)
                Predicted maliciousness score.
                Positive scores are maliciousness scores.
                A score of 0 means we found a match that was not malicious.
                Special cases:

                * -1: Not confident enough for prediction
                * -2: Label not in training
                * -3: Closest cluster > epsilon

In [17]:
# option1 Load results from CSV
label_filename="result/"+ test_filename.replace(".csv", "-onlyScoreLabel.csv")
final_scores, final_labels = load_anomaly_results(label_filename)


Loading anomaly results from result/AIA-201-225.ecar-onlyScoreLabel.csv
Successfully loaded 23189497 records
Anomaly distribution: 7538211.0 anomalies, 15651286.0 normal


In [38]:
# # option2 direct use results from memory
# final_scores=train_seq_scores
# final_labels=train_seq_labels

In [29]:
events_train  = events_train.to(args.device)
context_train  = context_train.to(args.device)

3


In [30]:
########################################################################
#                             Interpreter                            #
########################################################################
# args.load_interpreter="saved_Interpreter_model-2" # None   # args.save_interpreter

# Load the interpreter, if necessary
if args.load_interpreter:
    interpreter = Interpreter.load(
        args.load_interpreter,
        context_builder = context_builder,
    )
# Otherwise create a new Interpreter
else:
    # Create Interpreter
    interpreter = Interpreter(
        context_builder = context_builder,
        features        = args.events,
        eps             = args.epsilon,
        min_samples     = args.min_samples,
        threshold       = args.confidence,
    )



In [31]:
# Cluster samples with the interpreter
clusters = interpreter.cluster(
    X          = context_train,  # context,    #
    y          = events_train.reshape(-1, 1),   #  events.reshape(-1, 1),  #
    iterations = 3,  # 100,
    batch_size = 1024,
    verbose    = args.silent,  # not args.silent,
)
# # Save clusters, if necessary
# if args.save_clusters:

#     # Set labels to -1 if no labels were provided
#     if seq_label is None:
#         seq_label = np.full(clusters.shape[0], -1, dtype=int)

#     # Save to file
#     pd.DataFrame({
#         'clusters': clusters,
#         'labels'  : seq_label,
#     }).to_csv(args.save_clusters, index=False)

Clustering: 100%|█████████████████████████████████| 2/2 [01:01<00:00, 30.70s/it]


In [39]:
# in case of final_scores.dim>1
if len(final_scores.shape)>0:
    final_scores=final_scores[:,0]

print(final_scores.shape)

(3635009,)


In [40]:
# Compute scores for each cluster based on individual labels per sequence
scores = interpreter.score_clusters(
    scores   = final_scores, # train_seq_labels   # Labels used to compute score (either as loaded by Preprocessor, or put your own labels here)
    strategy = "min",        # Strategy to use for scoring (one of "max", "min", "avg")
    NO_SCORE = -1,           # Any sequence with this score will be ignored in the strategy.
                                # If assigned a cluster, the sequence will inherit the cluster score.
                                # If the sequence is not present in a cluster, it will receive a score of NO_SCORE.
)

# Assign scores to clusters in interpreter
# Note that all sequences should be given a score and each sequence in the
# same cluster should have the same score.
interpreter.score(
    scores  = scores, # Scores to assign to sequences
    verbose = True,   # If True, prints progress
)


# Save the interpreter, if necessary
# args.save_interpreter=None
args.save_interpreter="saved_model/saved_Interpreter_model-2-2-8858-"+test_filename
if args.save_interpreter:
    interpreter.save(args.save_interpreter)

Scoring: 100%|████████████████████████████████████| 2/2 [00:17<00:00,  8.82s/it]


In [41]:
def process_large_data_Interp_predict(big_contexts, big_events, args, model=None):
    chunk_size = args.dataloader_chunk_size    // 2
    total_size = len(big_events)
    num_chunks = (total_size + chunk_size - 1) // chunk_size

    temp_dir = tempfile.mkdtemp()  # Creates a temporary directory
    result1_paths, result2_paths = [], []

    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total_size)
        print(f"Processing chunk {i+1}/{num_chunks} (indices {start_idx} to {end_idx})")

        chunk_contexts = big_contexts[start_idx:end_idx]
        chunk_events = big_events[start_idx:end_idx]

        # with torch.no_grad():
        chunk_pred, chunk_index = model.predict(
            X=chunk_contexts, 
            y=chunk_events.reshape(-1, 1),
            iterations = 10,  # 100,                        # Number of iterations to use for attention query, in paper this was 100
            batch_size = 1024,                       # Batch size to use for attention query, used to limit CUDA memory usage
            verbose    = True,                       # If True, prints progress
        )

        # Save chunk results as .pt files
        path1 = os.path.join(temp_dir, f"chunk_pred_{i}.pt")
        path2 = os.path.join(temp_dir, f"chunk_index_{i}.pt")
        torch.save(chunk_pred, path1)
        torch.save(chunk_index, path2)
        result1_paths.append(path1)
        result2_paths.append(path2)

        # Clear memory
        del chunk_contexts, chunk_events, chunk_pred, chunk_index
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Load and concatenate all results
    all_result1 = [torch.load(p, weights_only=False) for p in result1_paths]
    all_result2 = [torch.load(p, weights_only=False) for p in result2_paths]

    final_result1 = np.concatenate(all_result1, axis=0)
    final_result2 = np.concatenate(all_result2, axis=0)

    # Clean up temporary files
    for p in result1_paths + result2_paths:
        os.remove(p)
    os.rmdir(temp_dir)

    return final_result1, final_result2


In [42]:
def threshold_search(y_true, y_proba):
    # search threshold for Accuracy
    best_threshold = 0
    best_score = 0
    for rate in np.arange(0.01,1, 0.01):
        threshold=np.quantile(y_proba,rate)
        y_pred=y_proba > threshold
        #score=metrics.f1_score(y_true, y_pred, average='weighted') 
        metric_report=classification_report(y_true, y_pred,output_dict=True)       
        try :
            f1_positive=metric_report['1.0']['f1-score']
        except:
            f1_positive=0
        if metric_report['accuracy'] >best_score and f1_positive!=0 : 
            best_threshold = threshold
            best_score = metric_report['accuracy']   #metric_report['macro avg']['f1-score']
    return best_score, best_threshold

In [ ]:
########################################################################
#                       Semi-automatic analysis                      #
########################################################################

#Compute predicted scores
if len(events_test)>args.dataloader_chunk_size * 3:
    pred, indexIntraining= process_large_data_Interp_predict(context_test, events_test.reshape(-1, 1), args, interpreter)
else:
    pred, indexIntraining = interpreter.predict(
        X          = context_test,               # Context to predict
        y          = events_test.reshape(-1, 1), # Events to predict, note that these should be of shape=(n_events, 1)
        iterations = 10,  # 100,                        # Number of iterations to use for attention query, in paper this was 100
        batch_size = 1024,                       # Batch size to use for attention query, used to limit CUDA memory usage
        verbose    = True,                       # If True, prints progress
    )

# # Check whether predictions can be saved
# if args.save_prediction is None:
#     raise ValueError(
#         "Please use --save-prediction CSV_FILE to specify a file to "
#         "save the predictions for each sequence."
#     )
# # Save to file
# pd.DataFrame({
#     'labels': prediction,
# }).to_csv(args.save_prediction, index=False)


In [ ]:

unique_vals = np.unique(pred)
# 
if not set(unique_vals).issubset({0, 1, -1,-2,-3}):
    mask_score = np.where((pred !=-1) & (pred !=-2) & (pred !=-3))[0]

    # option1 search threshold in quantile()
    m_score, m_threshold=threshold_search(labels_test_binary[mask_score], pred[mask_score])
    print(f"\n Best Threshold = {m_threshold:.2f}")
    print(f"Accuracy    = {m_score:.4f}")
    prediction=pred
    prediction[mask_score]=(pred[mask_score] > m_threshold).astype(float)

    # option2 through rocauc
    bestthresh,bestfpr,bestauc,_= rocauc('beAnom',labels_test_binary[mask_score], pred[mask_score],plot=True)      
    prediction2=pred
    prediction2[mask_score]=(pred[mask_score] > bestthresh).astype(float)        
    print(f"prediction2   AUC-ROC: {bestauc:.4f}")
    print(f"\n prediction2   Best Threshold = {bestthresh:.2f}")
    print(f"\n  prediction2    Best FPR = {bestfpr:.2f}")  
    print(classification_report(labels_test_binary,prediction2,digits=4))

else:
    prediction=pred

# If labels were provided, print classification report
if labels_test_binary is not None:

    # Print classification report
    print("Classification report")
    print(classification_report(
        y_pred        = prediction,
        y_true        = labels_test_binary,
        digits        = 4,
        zero_division = 0,
    ))

    # Print confusion matrix
    print("Confusion matrix")
    all_labels = np.unique(labels_test_binary).tolist()
    print(confusion_report(
        y_pred       = prediction,
        y_true       = labels_test_binary,
        labels       = [-3, -2, -1] + all_labels,
        target_names = ['LOW CONFIDENCE', 'NOT IN TRAIN', 'LOW EPS'] + all_labels,
        skip_x       = all_labels,
        skip_y       = ['LOW CONFIDENCE', 'NOT IN TRAIN', 'LOW EPS']
    ))

#Compute the accuracy
mask_p_n = np.where((prediction ==0) | (prediction ==1))[0]
result_predicted = prediction[mask_p_n]
seq_label_mask = labels_test_binary[mask_p_n]

report_class=classification_report(seq_label_mask,result_predicted,digits=4,output_dict=True)
print(f"weighted F1: {report_class['weighted avg']['f1-score']}")
print(f"macro F1: {str(report_class['macro avg']['f1-score'])}")

In [ ]:
# Print classification report on prediction2
print("Classification report")
print(classification_report(
    y_pred        = prediction2,
    y_true        = labels_test_binary,
    digits        = 4,
    zero_division = 0,
))

# Print confusion matrix
print("Confusion matrix")
all_labels = np.unique(labels_test_binary).tolist()
print(confusion_report(
    y_pred       = prediction2,
    y_true       = labels_test_binary,
    labels       = [-3, -2, -1] + all_labels,
    target_names = ['LOW CONFIDENCE', 'NOT IN TRAIN', 'LOW EPS'] + all_labels,
    skip_x       = all_labels,
    skip_y       = ['LOW CONFIDENCE', 'NOT IN TRAIN', 'LOW EPS']
))

In [ ]:
# filter rate= len( mask_score or mask_p_n )/len(prediction)
print(len(mask_score)==len(mask_p_n)) # true
filter_r=len(mask_score)/len(prediction))
print(filter_r)

In [ ]:
# Bundle metrics
metrics_dic = { 
    "datafile":test_filename,
    "modelname":'Interpreter',
    "mode": args.mode,
    "AUCROC": f"{bestauc:.4f}",
    "weighted_F1": f"{ report_class['weighted avg']['f1-score']:.4f}",
    "macro_F1": f"{report_class['macro avg']['f1-score']:.4f}",
    "FPR": f"{bestfpr:.4f}",   # 0 is the best
    "threshold": f"{bestthresh:.2f}",
    "filterrate":f"{filter_r:.2f}",
}
log_metrics_to_csv(metrics=metrics_dic,csv_path="result/Interp_test_metrics.csv")


In [ ]:
'''find out the origin record, with label, score, clusers for similar seqs'''
print( X_train.iloc[indexIntraining].iloc[0,:] )

In [ ]:
 # result = cls(
 #            context_builder= dictionary.get('context_builder'),
 #            features       = dictionary.get('features') ,
 #            eps            = dictionary.get('eps'),
 #            min_samples    = dictionary.get('min_samples'),
 #            threshold      = dictionary.get('threshold'),
 #        )

 #        result.clusters = dictionary.get('clusters')
 #        result.vectors  = dictionary.get('vectors')
 #        result.events   = dictionary.get('events')
 #        result.tree     = dictionary.get('tree')
 #        result.labels   = dictionary.get('labels')

indices=np.where(interpreter.clusters==interpreter.clusters[0])
print(  interpreter.clusters[indices] )
print(indices)

# X_train.iloc[indices]

In [ ]:
# vanilla BERT

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
# import torch

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Sample log event sequence
# log_sequence = "error failed connection timeout"
tokens = tokenizer(context_train_q, return_tensors="pt", truncation=True, padding=True)

# Forward pass (inference)
with torch.no_grad():
    output = model(**tokens)
print(output.logits.argmax().item())  # 0 (normal) or 1 (anomalous)


In [ ]:
# # LogBERT

# from transformers import BertTokenizer, BertForMaskedLM
# import torch

# # Load LogBERT tokenizer and model
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# # Example log sequence
# log_seq = "error failed connection timeout"

# # Mask a token (simulate missing event)
# tokens = tokenizer(log_seq, return_tensors="pt")
# masked_tokens = tokens.input_ids.clone()
# masked_tokens[0, 2] = tokenizer.mask_token_id  # Masking one word

# # Get predictions for the masked token
# with torch.no_grad():
#     outputs = model(input_ids=masked_tokens, attention_mask=tokens.attention_mask)
#     mask_logits = outputs.logits[:, 2, :]  # Get masked token predictions
#     anomaly_score = -mask_logits.max().item()  # Higher = more anomalous

# print(f"Anomaly Score: {anomaly_score}")



output event sequences with label from BERT, and attention vector from ContextBuilder.
trained ContextBuilder which will used to test.